# Phase 1: Training Pipeline — ResNet-50 Fine-tuning

**Task**: Conifer Family Classification (Task A)

This notebook trains a ResNet-50 model using fine-tuning with differential learning rates on 4 conifer families.

**Strategy** (per WoodID_plan.md):
- Backbone LR = 1e-5 (1/10 of head LR) — preserves pretrained ImageNet features
- Head LR = 1e-4
- 50 epochs, cosine annealing, label smoothing 0.1

---

## Contents

1. [Setup & Root Detection](#1-setup)  
2. [Install Dependencies](#2-install-dependencies)  
3. [Download Dataset from HuggingFace](#3-download-dataset)  
4. [Verify Dataset](#4-verify-dataset)  
5. [Create Dataloaders](#5-create-dataloaders)  
6. [Create & Test Model](#6-create-test-model)  
7. [Fine-tune ResNet-50](#7-fine-tune)  
8. [Evaluate on Test Set](#8-evaluate)

## 1. Setup & Root Detection <a id='1-setup'></a>

Detects whether running on Colab or local machine, and sets `ROOT` accordingly.

- **Colab**: `ROOT = /content/woodbench`
- **Local**: `ROOT = <repo-root>` (auto-detected from notebook location)

In [1]:
!pip install huggingface_hub
from google.colab import userdata
import os
from huggingface_hub import login
from huggingface_hub import snapshot_download

hftoken = userdata.get('HF_key')
login(token= hftoken)

os.environ["GH_TOKEN"] = userdata.get('GH_TOKEN')

!git clone https://$GH_TOKEN@github.com/ManifoldsTeam/woodbench.git

Cloning into 'woodbench'...
remote: Enumerating objects: 306, done.
remote: Counting objects: 100% (306/306), done.
remote: Compressing objects: 100% (207/207), done.
remote: Total 306 (delta 91), reused 277 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (306/306), 8.08 MiB | 31.80 MiB/s, done.
Resolving deltas: 100% (91/91), done.


In [2]:
import sys
from pathlib import Path

# Auto-detect ROOT: Colab uses /content/, local uses repo root
if Path('/content').exists():
    # Running on Colab
    ROOT = Path('/content/woodbench')
    print(f"[Colab] ROOT = {ROOT}")
else:
    # Running locally — infer repo root from notebook location
    # notebooks/phase 1/training_pipeline.ipynb → repo root
    ROOT = Path(__file__).parent.parent if '__file__' in dir() else Path.cwd()
    print(f"[Local] ROOT = {ROOT}")

# Add repo root to sys.path so `from src import ...` works
sys.path.insert(0, str(ROOT))

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

[Colab] ROOT = /content/woodbench
PyTorch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## 2. Install Dependencies <a id='2-install-dependencies'></a>

Only run this cell on Colab. Skip if already installed locally.

In [3]:
# Only run on Colab
# if Path('/content').exists():
#     !pip install -q -r {ROOT}/requirements.txt

## Mount Drive:

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Download Dataset from HuggingFace <a id='3-download-dataset'></a>

Downloads `IC4SD-Wood-Conifer20v1` from HuggingFace Hub to `/data/split/`.
Only downloads if data is not already present.

In [5]:
from pathlib import Path

DATA_DIR = ROOT / "data" / "split"

if not DATA_DIR.exists():
    from huggingface_hub import snapshot_download
    print("Downloading dataset from HuggingFace...")
    snapshot_download(
        repo_id="ny0g507h3p/woodbench_A",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
    )
else:
    print(f"Dataset already exists at {DATA_DIR}")

print(f"DATA_DIR = {DATA_DIR}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

DATA_DIR = /content/woodbench/data/split


In [6]:
%cd woodbench

/content/woodbench


In [7]:
!git checkout feat/task-A

Branch 'feat/task-A' set up to track remote branch 'feat/task-A' from 'origin'.
Switched to a new branch 'feat/task-A'


In [8]:
!ls -a data/split

.	dataset_manifest.csv  quality_file_sizes.png   split_metadata.json  val
..	.gitattributes	      quality_image_sizes.png  test
.cache	qa_summary.json       split_distribution.png   train


## 4. Verify Dataset <a id='4-verify-dataset'></a>

In [9]:
from src.data import ConiferDataset

# Check structure
print("Dataset structure:")
for split in ['train', 'val', 'test']:
    split_path = DATA_DIR / split
    if split_path.exists():
        families = [d.name for d in split_path.iterdir() if d.is_dir()]
        print(f"  {split}/: {len(families)} families")

# Load train set and show distribution
train_ds = ConiferDataset(DATA_DIR, split='train', task='taskA')
print(f"\nTotal training samples: {len(train_ds)}")
print(f"Classes: {train_ds.classes}")
print(f"\nClass distribution:")
for family, count in train_ds.get_class_distribution().items():
    print(f"  {family}: {count} ({count/len(train_ds)*100:.1f}%)")

Dataset structure:
  train/: 4 families
  val/: 4 families
  test/: 4 families

Total training samples: 4487
Classes: ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']

Class distribution:
  Cupressaceae: 2398 (53.4%)
  Pinaceae: 1666 (37.1%)
  Podocarpaceae: 217 (4.8%)
  Taxaceae: 206 (4.6%)


## 5. Create Dataloaders <a id='5-create-dataloaders'></a>

In [10]:
from src.data import get_dataloaders

BATCH_SIZE = 64
IMG_SIZE = 224
NUM_WORKERS = 4  # Reduce to 2 on Colab if needed

train_loader, val_loader, test_loader = get_dataloaders(
    data_root=DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    img_size=IMG_SIZE,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Verify a batch
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")

Train batches: 70
Val batches:   15
Test batches:  16

Batch shape: torch.Size([64, 3, 224, 224])
Labels shape: torch.Size([64])


## 6. Create & Test Model <a id='6-create-test-model'></a>

In [11]:
from src.models import create_model, BACKBONE_NAMES

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Check available backbones
print("\nAvailable backbones:")
for name, timm_name in BACKBONE_NAMES.items():
    print(f"  {name}: {timm_name}")

# Create model (unfrozen for fine-tuning)
model = create_model('resnet50', num_classes=4, pretrained=True, freeze_backbone=False)
model = model.to(device)

# Test forward pass
outputs = model(images.to(device))
print(f"\nOutput shape: {outputs.shape}  (expected: ({images.shape[0]}, 4))")

# Verify trainable parameters
trainable = {n for n, p in model.named_parameters() if p.requires_grad}
frozen   = {n for n, p in model.named_parameters() if not p.requires_grad}
print(f"\nTrainable params: {len(trainable)}")
print(f"Frozen params:    {len(frozen)}")

Using device: cuda

Available backbones:
  resnet50: resnet50
  efficientnet_b3: tf_efficientnet_b3
  mobilenetv3_large: mobilenetv3_large_100
  convnext_tiny: convnext_tiny
  swin_tiny: swin_tiny_patch4_window7_224


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]


Output shape: torch.Size([64, 4])  (expected: (64, 4))

Trainable params: 161
Frozen params:    0


## 7. Fine-tune ResNet-50 <a id='7-fine-tune'></a>

Backbone LR = 1e-5, Head LR = 1e-4, 50 epochs, cosine annealing, AMP.

In [12]:
DRIVE_ROOT = Path("/content/drive/MyDrive")
EXP_DIR = DRIVE_ROOT / "woodbench_experiments" / "taskA" / "resnet50"

EXP_DIR.mkdir(parents=True, exist_ok=True)

print(EXP_DIR)

/content/drive/MyDrive/woodbench_experiments/taskA/resnet50


In [13]:
from src.train import TrainingEngine, create_optimizer, create_scheduler, create_loss_function
from src.utils import set_seed

set_seed(42)

# Re-create model (fresh init to start from pretrained)
model = create_model('resnet50', num_classes=4, pretrained=True, freeze_backbone=False)
model = model.to(device)

# Differential learning rates
HEAD_LR    = 1e-4
BACKBONE_LR = 1e-5
optimizer = create_optimizer(
    model,
    lr=HEAD_LR,
    weight_decay=1e-4,
    backbone_lr=BACKBONE_LR,
)

# Verify param groups
for pg in optimizer.param_groups:
    print(f"  [{pg['name']}] lr={pg['lr']:.2e}, {len(pg['params'])} tensors")

scheduler = create_scheduler(optimizer, epochs=50, scheduler_type='cosine')
criterion = create_loss_function(loss_type='label_smoothing', label_smoothing=0.1)

engine = TrainingEngine(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device,
    use_amp=True,
)

EXP_DIR = DRIVE_ROOT / "woodbench_experiments" / "taskA" / "resnet50"
EXP_DIR.mkdir(parents=True, exist_ok=True)
print(EXP_DIR)

print("\nStarting fine-tuning (50 epochs)...")
results = engine.fit(
    epochs=50,
    output_dir=EXP_DIR,
    name='resnet50_finetune',
    save_best_only=False,
    verbose=True,
)

print(f"\nBest val accuracy: {results['best_val_acc']:.2f}%")

  [head] lr=1.00e-04, 2 tensors
  [backbone] lr=1.00e-05, 159 tensors
/content/drive/MyDrive/woodbench_experiments/taskA/resnet50

Starting fine-tuning (50 epochs)...


/content/woodbench/src/train/trainer.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if use_amp else None
/content/woodbench/src/train/trainer.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/woodbench/src/train/trainer.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.use_amp):


Epoch 1/50 | Train Loss: 1.2047, Acc: 52.17% | Val Loss: 1.1240, Acc: 53.40% | LR: 9.99e-05
Epoch 2/50 | Train Loss: 1.0695, Acc: 53.75% | Val Loss: 1.0503, Acc: 53.51% | LR: 9.96e-05
Epoch 3/50 | Train Loss: 1.0143, Acc: 59.75% | Val Loss: 0.9939, Acc: 62.72% | LR: 9.91e-05
Epoch 4/50 | Train Loss: 0.9626, Acc: 67.59% | Val Loss: 0.9395, Acc: 68.59% | LR: 9.84e-05
Epoch 5/50 | Train Loss: 0.9097, Acc: 72.01% | Val Loss: 0.8868, Acc: 72.67% | LR: 9.76e-05
Epoch 6/50 | Train Loss: 0.8586, Acc: 76.50% | Val Loss: 0.8336, Acc: 76.96% | LR: 9.65e-05
Epoch 7/50 | Train Loss: 0.8110, Acc: 79.08% | Val Loss: 0.7852, Acc: 80.42% | LR: 9.52e-05
Epoch 8/50 | Train Loss: 0.7712, Acc: 80.29% | Val Loss: 0.7458, Acc: 83.14% | LR: 9.38e-05
Epoch 9/50 | Train Loss: 0.7361, Acc: 82.23% | Val Loss: 0.7087, Acc: 83.56% | LR: 9.22e-05
Epoch 10/50 | Train Loss: 0.7081, Acc: 82.88% | Val Loss: 0.6775, Acc: 85.65% | LR: 9.05e-05
Epoch 11/50 | Train Loss: 0.6751, Acc: 84.42% | Val Loss: 0.6464, Acc: 86.39% |

In [15]:
!ls -a /content/drive/MyDrive/woodbench_experiments/taskA/resnet50/

resnet50_finetune_epoch10.pth  resnet50_finetune_epoch34.pth
resnet50_finetune_epoch11.pth  resnet50_finetune_epoch35.pth
resnet50_finetune_epoch12.pth  resnet50_finetune_epoch36.pth
resnet50_finetune_epoch13.pth  resnet50_finetune_epoch37.pth
resnet50_finetune_epoch14.pth  resnet50_finetune_epoch38.pth
resnet50_finetune_epoch15.pth  resnet50_finetune_epoch39.pth
resnet50_finetune_epoch16.pth  resnet50_finetune_epoch3.pth
resnet50_finetune_epoch17.pth  resnet50_finetune_epoch40.pth
resnet50_finetune_epoch18.pth  resnet50_finetune_epoch41.pth
resnet50_finetune_epoch19.pth  resnet50_finetune_epoch42.pth
resnet50_finetune_epoch1.pth   resnet50_finetune_epoch43.pth
resnet50_finetune_epoch20.pth  resnet50_finetune_epoch44.pth
resnet50_finetune_epoch21.pth  resnet50_finetune_epoch45.pth
resnet50_finetune_epoch22.pth  resnet50_finetune_epoch46.pth
resnet50_finetune_epoch23.pth  resnet50_finetune_epoch47.pth
resnet50_finetune_epoch24.pth  resnet50_finetune_epoch48.pth
resnet50_finetune_epoch25

## 8. Evaluate on Test Set <a id='8-evaluate'></a>

Loads the best checkpoint and evaluates on the held-out test set.

In [16]:
from src.eval import compute_metrics, format_metrics, save_confusion_matrix

CLASS_NAMES = ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']

# Load best checkpoint
checkpoint_path = EXP_DIR / "resnet50_finetune_epoch48.pth"
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded checkpoint from epoch {checkpoint['epoch']+1}")

# Evaluate
metrics = compute_metrics(
    model=model,
    data_loader=test_loader,
    device=device,
    num_classes=4,
    class_names=CLASS_NAMES,
)

print(format_metrics(metrics, CLASS_NAMES))

# Save confusion matrix
save_confusion_matrix(
    cm=metrics['confusion_matrix'],
    class_names=CLASS_NAMES,
    output_path=EXP_DIR / "confusion_matrix.png",
    title="Confusion Matrix — ResNet-50 Fine-tune (Test Set)",
)
print(f"Confusion matrix saved to {EXP_DIR / 'confusion_matrix.png'}")

Loaded checkpoint from epoch 48
EVALUATION RESULTS
Accuracy:      96.64%
Precision:     94.03%
Recall:        88.21%
F1-macro:      90.23%

Per-Class F1 Scores:
  Cupressaceae        : 97.45%
  Pinaceae            : 98.36%
  Podocarpaceae       : 94.85%
  Taxaceae            : 70.27%
Confusion matrix saved to /content/drive/MyDrive/woodbench_experiments/taskA/resnet50/confusion_matrix.png
